In [1]:
import pandas as pd
df_train = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/TACRED/Train/MINT/step0_TACRED_full_phrase.csv')
print(df_train.shape) 
df_train.head(2)

(1900, 20)


,id,relation,text,subject,subject_type,object,object_type,subj_start,subj_end,obj_start,obj_end,docid,token,subj_type,obj_type,stanford_pos,stanford_ner,stanford_head,stanford_deprel,gpt4_phrase
0,61b3aa9e36d5d8427e8e,org:alternate_names,"DC : Nov 30th thru Dec 2nd , Servicemembers Un...",Servicemembers United,ORGANIZATION,Servicemembers Legal Defense Network,ORGANIZATION,8,9,28,31,eng-WL-11-174611-12982547,"['DC', ':', 'Nov', '30th', 'thru', 'Dec', '2nd...",ORGANIZATION,ORGANIZATION,"['NN', ':', 'NNP', 'JJ', 'IN', 'NNP', 'JJ', ',...","['O', 'O', 'DATE', 'DATE', 'O', 'DATE', 'DATE'...","[0, 1, 18, 3, 7, 7, 4, 7, 10, 7, 13, 13, 7, 15...","['ROOT', 'punct', 'nsubj', 'amod', 'case', 'co...",Partnership
1,61b3aa9e368a0e3dfbd2,org:alternate_names,With the tally completed in more than half the...,All Basotho Convention,ORGANIZATION,ABC,ORGANIZATION,29,31,33,33,AFP_ENG_20070219.0359.LDC2009T13,"['With', 'the', 'tally', 'completed', 'in', 'm...",ORGANIZATION,ORGANIZATION,"['IN', 'DT', 'NN', 'VBN', 'IN', 'JJR', 'IN', '...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","[3, 3, 22, 3, 11, 8, 6, 11, 11, 11, 4, 13, 11,...","['case', 'det', 'nmod', 'acl', 'case', 'advmod...",abbreviation


In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize


texts = df_train["gpt4_phrase"].fillna("").astype(str).tolist()



print("num texts:", len(texts))


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("embedding shape:", embeddings.shape)


def estimate_k_by_zhang2021(embeddings, k0, random_state=42):
    """
    Zhang et al. (2021a) 방식의 unknown K 추정.

    Args:
        embeddings   : clustering할 embedding matrix, shape = (N, dim)
        k0           : over-clustering에 사용할 큰 cluster 개수
        random_state : random seed

    Returns:
        estimated_k  : 추정된 cluster 개수
        labels_k0    : K0로 clustering한 결과
        cluster_info : cluster별 size 정보
    """

    n = len(embeddings)

    kmeans = KMeans(
        n_clusters=k0,
        random_state=random_state,
        n_init="auto"
    )

    labels_k0 = kmeans.fit_predict(embeddings)

    cluster_sizes = pd.Series(labels_k0).value_counts().sort_index()

    threshold = n / k0

    valid_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
    estimated_k = len(valid_clusters)

    cluster_info = pd.DataFrame({
        "cluster_id": cluster_sizes.index,
        "size": cluster_sizes.values,
        "threshold": threshold,
        "is_valid": cluster_sizes.values >= threshold
    })

    return estimated_k, labels_k0, cluster_info

num texts: 1900


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

embedding shape: (1900, 384)


In [3]:
k0 = 300

estimated_k, labels_k0, cluster_info = estimate_k_by_zhang2021(
    embeddings=embeddings,
    k0=k0,
    random_state=42
)

print("Estimated K:", estimated_k)
cluster_info

Estimated K: 93


,cluster_id,size,threshold,is_valid
0,0,5,6.333333,False
1,1,1,6.333333,False
2,2,12,6.333333,True
3,3,8,6.333333,True
4,4,66,6.333333,True
...,...,...,...,...
295,295,1,6.333333,False
296,296,3,6.333333,False
297,297,1,6.333333,False
298,298,3,6.333333,False


In [4]:
kmeans = KMeans(
    n_clusters=estimated_k,
    random_state=42,
    n_init="auto"
)

df_train["cluster_unknown_k"] = kmeans.fit_predict(embeddings)


# Evaluation

In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment


def evaluate_clustering(df: pd.DataFrame, pred_col: str, gold_col: str = "relation"):

    
    non_na_df = df[~df[pred_col].isna()].copy()

    if len(non_na_df) == 0:
        raise ValueError(f"{pred_col}에 유효한 값이 없습니다.")

    y_true = non_na_df[gold_col].astype(str).values
    y_pred = non_na_df[pred_col].astype(str).values

    
    overall_purity = (
        non_na_df
        .groupby(pred_col)[gold_col]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(non_na_df)
    )

    
    ari = adjusted_rand_score(y_true, y_pred)

    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)

    
    def b3_precision_recall_f1(y_true, y_pred):

        true_groups = {}
        pred_groups = {}

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            true_groups.setdefault(t, set()).add(i)
            pred_groups.setdefault(p, set()).add(i)

        P_sum, R_sum = 0.0, 0.0
        n = len(y_true)

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            T = true_groups[t]
            C = pred_groups[p]
            inter = len(T & C)

            P_sum += inter / len(C) if len(C) > 0 else 0.0
            R_sum += inter / len(T) if len(T) > 0 else 0.0

        P = P_sum / n
        R = R_sum / n
        F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0

        return P, R, F1

    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    
    print(f"Purity : {overall_purity:.4f}")
    print(f"ACC    : {hungarian_acc:.4f}")
    
    
    print(f"B3-F1  : {b3_f1:.4f}")
    
    
    print(f"V-F1   : {v_f1:.4f}")
    print(f"ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }

In [6]:


metrics = evaluate_clustering(
    df=df_train,
    pred_col="cluster_unknown_k",
    gold_col="relation"
)

metrics

Purity : 0.5495
ACC    : 0.3616
B3-F1  : 0.3215
V-F1   : 0.6109
ARI    : 0.2888


{'Purity': 0.5494736842105263,
 'ARI': 0.2887962765371946,
 'V-hom': 0.6713078480666846,
 'V-com': 0.5605171059273509,
 'V-F1': 0.6109301990751485,
 'B3-P': 0.44647489010643765,
 'B3-R': 0.2512210526315795,
 'B3-F1': 0.3215265705170519,
 'ACC': 0.36157894736842106}

# Test set Evaluation

In [7]:
test_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/TACRED/Test/TACRED_test_phrase.csv')
print(test_df.shape)
test_df.head(2)

(760, 20)


,id,relation,text,subject,subject_type,object,object_type,subj_start,subj_end,obj_start,obj_end,docid,token,subj_type,obj_type,stanford_pos,stanford_ner,stanford_head,stanford_deprel,gpt4_phrase
0,e7798a9e36681e051e7f,org:alternate_names,The China Banking Regulatory Commission -LRB- ...,China Banking Regulatory Commission,ORGANIZATION,CBRC,ORGANIZATION,1,4,6,6,XIN_ENG_20081206.0210.LDC2009T13,"['The', 'China', 'Banking', 'Regulatory', 'Com...",ORGANIZATION,ORGANIZATION,"['DT', 'NNP', 'NNP', 'NNP', 'NNP', '-LRB-', 'N...","['O', 'ORGANIZATION', 'ORGANIZATION', 'ORGANIZ...","[5, 5, 5, 5, 9, 7, 5, 7, 0, 13, 13, 13, 15, 15...","['det', 'compound', 'compound', 'compound', 'n...",Acronym_Definition
1,098f6a9e36f693f19430,org:alternate_names,TPI founder Prachai Leophairatana got three ye...,TPIPL,ORGANIZATION,TPI,ORGANIZATION,18,18,0,0,AFP_ENG_20071203.0448.LDC2009T13,"['TPI', 'founder', 'Prachai', 'Leophairatana',...",ORGANIZATION,ORGANIZATION,"['NNP', 'NN', 'NNP', 'NNP', 'VBD', 'CD', 'NNS'...","['ORGANIZATION', 'O', 'PERSON', 'PERSON', 'O',...","[4, 4, 4, 5, 0, 7, 5, 7, 7, 11, 5, 13, 11, 16,...","['compound', 'compound', 'compound', 'nsubj', ...",subsidiary_of


In [8]:



if "gpt4_phrase" not in test_df.columns:
    raise ValueError("test_df에 'gpt4_phrase' 컬럼이 없습니다.")

if "relation" not in test_df.columns:
    raise ValueError("test_df에 gold label 컬럼 'relation'이 없습니다.")

test_texts = test_df["gpt4_phrase"].fillna("").astype(str).tolist()

print("num test instances:", len(test_texts))

test_embeddings = model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("test_embeddings shape:", test_embeddings.shape)

test_df["cluster_unknown_k"] = kmeans.predict(test_embeddings)

print(test_df[[
    "relation",
    "gpt4_phrase",
    "cluster_unknown_k"
]].head(10))

num test instances: 760


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

test_embeddings shape: (760, 384)
              relation                                        gpt4_phrase  \
0  org:alternate_names                                 Acronym_Definition   
1  org:alternate_names                                      subsidiary_of   
2  org:alternate_names                                         located_in   
3  org:alternate_names                                              alias   
4  org:alternate_names  agency_is_identified_as_Federal_Motor_Carrier_...   
5  org:alternate_names               CorporateAffiliationBetweenCompanies   
6  org:alternate_names                                 Acronym_Definition   
7  org:alternate_names                                    abbreviation_of   
8  org:alternate_names                                              alias   
9  org:alternate_names                                 organization_alias   

   cluster_unknown_k  
0                 41  
1                 31  
2                  7  
3                 12  
4  

In [9]:
metrics_kmeans_test = evaluate_clustering(
    df=test_df,
    pred_col="cluster_unknown_k",
    gold_col="relation"
)

metrics_kmeans_test

Purity : 0.5842
ACC    : 0.3842
B3-F1  : 0.3634
V-F1   : 0.6649
ARI    : 0.2991


{'Purity': 0.5842105263157895,
 'ARI': 0.2990540760829641,
 'V-hom': 0.7238897374625526,
 'V-com': 0.6148141911030572,
 'V-F1': 0.6649083100289587,
 'B3-P': 0.496933325570833,
 'B3-R': 0.2864473684210529,
 'B3-F1': 0.36341269189348135,
 'ACC': 0.38421052631578945}

In [10]:
Purity : 0.5842
ACC    : 0.3842
B3-F1  : 0.3634
V-F1   : 0.6649
ARI    : 0.2991

SyntaxError: illegal target for annotation (570740961.py, line 3)